In [1]:
import json
from collections import Counter
import statistics
import pandas as pd

# 경로를 여기서 직접 지정
path = "../../results/phase1_easy/single/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)
len(data)

df = pd.DataFrame(data)
df.head()

,dataset,task_id,entry_point,method,model_name,attempt_idx,status,passed,timeout,raw_output,generated_code,error_type,error_message,latency_sec
0,humaneval,HumanEval/0,has_close_elements,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,for i in range(len(numbers)):\n for...,from typing import List\n\n\ndef has_close_ele...,None,None,1.012781
1,humaneval,HumanEval/1,separate_paren_groups,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,result = []\n current_group = ''\n d...,from typing import List\n\n\ndef separate_pare...,None,None,1.404479
2,humaneval,HumanEval/2,truncate_number,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,fail,False,False,return number - int(number)\n\n# Test case...,\n\ndef truncate_number(number: float) -> floa...,assertion_error,"Traceback (most recent call last):\n File ""/t...",1.264595
3,humaneval,HumanEval/3,below_zero,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,balance = 0\n for operation in operatio...,from typing import List\n\n\ndef below_zero(op...,None,None,0.667720
4,humaneval,HumanEval/4,mean_absolute_deviation,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,if not numbers:\n return 0.0\n\n ...,from typing import List\n\n\ndef mean_absolute...,None,None,4.236917


In [2]:
# 전체 요약
total = len(data)
passed = sum(1 for x in data if x["status"] == "pass")
timeout = sum(1 for x in data if x["status"] == "timeout")
failed = total - passed - timeout

print("=" * 60)
print(f"📂 File: {path}")
print("📊 Overall")
print(f"Total: {total}")
print(f"Pass: {passed}")
print(f"Fail: {failed}")
print(f"Timeout: {timeout}")
print(f"Pass@1: {passed / total:.4f}")

📂 File: ../../results/phase1_easy/single/details.jsonl
📊 Overall
Total: 164
Pass: 118
Fail: 46
Timeout: 0
Pass@1: 0.7195


In [3]:
# error distribution
error_counter = Counter(
    x["error_type"] for x in data if x["error_type"] is not None
)

print("\n📉 Error Type Distribution")
for k, v in error_counter.most_common():
    print(f"{k}: {v}")
    
latencies = [x["latency_sec"] for x in data if x["latency_sec"] is not None]

if latencies:
    print("\n⏱ Latency")
    print(f"Avg: {statistics.mean(latencies):.3f}s")
    print(f"Max: {max(latencies):.3f}s")


📉 Error Type Distribution
assertion_error: 28
syntax_error: 13
name_error: 3
type_error: 1
value_error: 1

⏱ Latency
Avg: 2.117s
Max: 6.614s


### 실패 샘플 보기

In [4]:
print("\n❌ Sample Failures (up to 3)")

failures = [x for x in data if x["status"] == "fail"][:3]

for f in failures:
    print("-" * 40)
    print(f"Task: {f['task_id']}")
    print(f"Error Type: {f['error_type']}")
    print(f"Error Msg: {f['error_message']}")


❌ Sample Failures (up to 3)
----------------------------------------
Task: HumanEval/2
Error Type: assertion_error
Error Msg: Traceback (most recent call last):
  File "/tmp/tmpfzonr3x_.py", line 17, in <module>
    assert truncate_number(9.99) == 0.99
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError

----------------------------------------
Task: HumanEval/10
Error Type: assertion_error
Error Msg: Traceback (most recent call last):
  File "/tmp/tmp6xgsdi15.py", line 41, in <module>
    check(make_palindrome)
  File "/tmp/tmp6xgsdi15.py", line 36, in check
    assert candidate('x') == 'x'
           ^^^^^^^^^^^^^^^^^^^^^
AssertionError

----------------------------------------
Task: HumanEval/26
Error Type: assertion_error
Error Msg: Traceback (most recent call last):
  File "/tmp/tmprw26qu20.py", line 26, in <module>
    check(remove_duplicates)
  File "/tmp/tmprw26qu20.py", line 24, in check
    assert candidate([1, 2, 3, 2, 4, 3, 5]) == [1, 4, 5]
           ^^^^^^^^^^^^^^^^^

In [5]:
failures = [x for x in data if x["status"] == "fail"][:2]

for f in failures:
    print("=" * 80)
    print(f"TASK: {f['task_id']}")
    print(f"ERROR: {f['error_type']}")
    print("\n--- GENERATED CODE (head) ---")
    print(f["generated_code"][:500])

TASK: HumanEval/2
ERROR: assertion_error

--- GENERATED CODE (head) ---


def truncate_number(number: float) -> float:
    """ Given a positive floating point number, it can be decomposed into
    and integer part (largest integer smaller than given number) and decimals
    (leftover part always smaller than 1).

    Return the decimal part of the number.
    >>> truncate_number(3.5)
    0.5
    """
    return number - int(number)

# Test cases to validate the correctness of the function
assert truncate_number(3.5) == 0.5
assert truncate_number(4.75) == 0.75
assert 
TASK: HumanEval/10
ERROR: assertion_error

--- GENERATED CODE (head) ---


def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Appen